# HSE DSBA Chatbot — RAG Retrieval Evaluation
Тестирование качества ретрива на 50 вопросах по программе ПАД

## 1. Импорты

In [53]:
import re
import os
import time
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

print("Imports loaded")

Imports loaded


## 2. Загрузка текста, EDA и чанкинг

In [54]:
with open('About-program.txt', 'r', encoding='utf-8') as f:
    text = f.read()

num_chars = len(text)
num_words = len(text.split())
num_lines = len(text.splitlines())

sections_for_eda = [s.strip() for s in re.split(r'-{5,}', text) if s.strip()]

empty_lines = 0
long_dash_lines = 0
double_spaces = text.count("  ")

for line in text.splitlines():
    if line.strip() == "":
        empty_lines = empty_lines + 1

    if "-----" in line:
        long_dash_lines = long_dash_lines + 1

eda_table = pd.DataFrame({
    "Metric": [
        "Characters",
        "Words",
        "Lines",
        "Sections",
        "Empty lines",
        "Lines with long dashes",
        "Double spaces"
    ],
    "Value": [
        num_chars,
        num_words,
        num_lines,
        len(sections_for_eda),
        empty_lines,
        long_dash_lines,
        double_spaces
    ]
})

display(eda_table)


def split_into_chunks(text, chunk_size=500, overlap=50):
    sections = re.split(r'-{5,}', text)
    chunks = []

    for section in sections:
        section = section.strip()

        if section == "":
            continue

        words = section.split()
        step = chunk_size - overlap

        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + chunk_size])

            if len(chunk) > 50:
                chunks.append(chunk)

    return chunks


chunks = split_into_chunks(text, chunk_size=500, overlap=50)

print("Chunker: fixed-size word chunker with overlap")
print("Chunk size: 500 words")
print("Overlap: 50 words")
print(f"Number of chunks: {len(chunks)}")
print()
print("First chunk:")
print(chunks[0][:500])

,Metric,Value
0,Characters,32113
1,Words,3777
2,Lines,606
3,Sections,64
4,Empty lines,66
5,Lines with long dashes,63
6,Double spaces,8


Chunker: fixed-size word chunker with overlap
Chunk size: 500 words
Overlap: 50 words
Number of chunks: 67

First chunk:
ВСЯ ПОСЛЕДУЮЩАЯ ИНФОРМАЦИЯ БУДЕТ АКТУАЛЬНАЯ НА 2024/2025 ГОД, НЕКОТОРЫЕ ДАННЫЕ БУДУТ АКТУАЛЬНЫ НА 2025/2026 УЧЕБНЫЙ ГОД: О программе: Целью программы является подготовка высококвалифицированных аналитиков и специалистов в области Data Scienсе, обладающих пониманием задач экономики и финансов, бизнеса и других прикладных областей, умеющих творчески применять свои знания и умения для успешного их решения. Основные задачи: Получение студентами хорошей математической подготовки, традиционной для рос


Датасет был собран вручную с сайта программы HSE DSBA / ПАД и сохранён в один текстовый файл. В нём есть разные типы информации: описание программы, поступление, FAQ, учебный план, скидки и карьерные возможности.

Текст неоднородный: часть информации написана обычными абзацами, часть — списками, часть — в формате вопросов и ответов. Так как данные были скопированы вручную, часть структуры сайта могла потеряться: например, таблицы, заголовки, ссылки или связь между разделами.

## 3. Golden set — 50 вопросов

In [55]:
golden_set = [
    ("Какова основная цель программы ПАД?", "подготовка высококвалифицированных аналитиков", "program_info"),
    ("В каком году была создана программа ПАД?", "2018", "program_info"),
    ("С каким британским университетом создана программа?", "LSE", "program_info"),
    ("На каком факультете реализуется программа ПАД?", "факультете компьютерных наук", "program_info"),
    ("В каком году ВШЭ и Яндекс открыли факультет компьютерных наук?", "2014", "program_info"),
    ("Какие специализации есть на программе?", "Анализ данных в бизнесе", "program_info"),
    ("На каком языке ведётся обучение на программе?", "английский", "program_info"),
    ("Когда был выпущен первый набор студентов ПАД?", "2022", "program_info"),

    ("Какие предметы ЕГЭ нужны для поступления на ПАД?", "Математика", "admission"),
    ("Какой минимальный балл ЕГЭ по математике?", "70", "admission"),
    ("Какой минимальный балл ЕГЭ по физике или информатике?", "60", "admission"),
    ("Какой минимальный балл ЕГЭ по русскому языку?", "60", "admission"),
    ("Сколько платных мест на программе ПАД?", "220", "admission"),
    ("Есть ли бюджетные места на программе ПАД?", "бюджетные места не предусмотрены", "admission"),
    ("Имеют ли победители олимпиад льготы при поступлении?", "олимпиад", "admission"),
    ("Сколько мест для иностранных студентов?", "10 платных мест", "admission"),
    ("Какие экзамены сдают иностранцы по отдельному конкурсу?", "Математика и Английский язык", "admission"),
    ("Какой минимальный балл по математике для иностранцев?", "75", "admission"),
    ("Какой минимальный балл по английскому для иностранцев?", "70", "admission"),
    ("Можно ли сдать вступительные экзамены дистанционно?", "дистанционно", "admission"),
    ("Какой экзамен нужен иностранцу с российским гражданством?", "русскому", "admission"),

    ("Какова стоимость обучения на 2025 год?", "1 млн", "tuition"),
    ("Какой коэффициент у балла ЕГЭ по математике при ранжировании?", "коэффициент 3", "tuition"),
    ("До какой даты нужно предоставить диплом олимпиады для скидки?", "22 июля", "tuition"),
    ("Когда публикуется список претендентов на скидку?", "25 июля", "tuition"),
    ("Какой максимальный размер скидки по результатам обучения?", "75%", "tuition"),
    ("Какой минимальный размер скидки по результатам обучения?", "10", "tuition"),
    ("Кому адресовать вопросы по скидкам?", "Шахвердян", "tuition"),
    ("Учитываются ли индивидуальные достижения при ранжировании?", "индивидуальные достижения", "tuition"),
    ("По каким олимпиадам учитываются достижения для скидки?", "математике, информатике, физике", "tuition"),

    ("На каком курсе начинается изучение Python?", "Программирование на Python", "curriculum"),
    ("На каком курсе изучается Машинное обучение?", "Машинное обучение", "curriculum"),
    ("На каком курсе изучаются Базы данных?", "Базы данных", "curriculum"),
    ("На каком курсе изучается Глубокое обучение?", "Глубокое обучение", "curriculum"),
    ("На каком курсе изучается Эконометрика?", "Эконометрика", "curriculum"),
    ("Является ли курс Линейная алгебра обязательным?", "Линейная алгебра", "curriculum"),
    ("На каком курсе изучается Компьютерное зрение?", "Компьютерное зрение", "curriculum"),
    ("На каком курсе изучается Анализ временных рядов?", "Анализ временных рядов", "curriculum"),

    ("Можно ли получить диплом LSE обучаясь на ПАД?", "приостановке сотрудничества", "faq"),
    ("Чем ПАД отличается от ПМИ?", "ПМИ", "faq"),
    ("Чем ПАД отличается от ЭиАД?", "ЭиАД", "faq"),
    ("Предоставляется ли отсрочка от армии студентам ПАД?", "отсрочка", "faq"),
    ("Трудно ли сразу учиться на английском?", "языковой подготовке", "faq"),
    ("Есть ли на ФКН военный учебный центр?", "Военный учебный центр", "faq"),
    ("Изменился ли учебный план после приостановки сотрудничества с LSE?", "изменений в учебном плане не произошло", "faq"),

    ("Когда на ПАД начинается практическая проектная работа?", "2-го курса", "career"),
    ("Какой размер стипендии Яндекса для студентов-бакалавров?", "30", "career"),
    ("Сколько студентов-бакалавров получают стипендию Яндекса?", "десять", "career"),
    ("В каких компаниях могут работать выпускники ПАД?", "Яндекс", "career"),
    ("Какую магистерскую программу открыл ФКН в партнёрстве со Сбером?", "Финансовые технологии", "career"),
]

golden_df = pd.DataFrame(golden_set, columns=["question", "keyword", "category"])

print(f"Golden set: {len(golden_df)} questions")
display(golden_df.head())

Golden set: 50 questions


,question,keyword,category
0,Какова основная цель программы ПАД?,подготовка высококвалифицированных аналитиков,program_info
1,В каком году была создана программа ПАД?,2018,program_info
2,С каким британским университетом создана программа?,LSE,program_info
3,На каком факультете реализуется программа ПАД?,факультете компьютерных наук,program_info
4,В каком году ВШЭ и Яндекс открыли факультет компьютерных наук?,2014,program_info


## 4. Анализ golden set

In [56]:
category_counts = golden_df["category"].value_counts().reset_index()
category_counts.columns = ["category", "count"]

display(category_counts)

,category,count
0,admission,13
1,tuition,9
2,program_info,8
3,curriculum,8
4,faq,7
5,career,5


В golden set вошли 50 вопросов по основным темам программы: информация о программе, поступление, стоимость обучения, учебный план, FAQ и карьерные возможности.

Большая часть вопросов является фактической: для них можно проверить, найден ли правильный фрагмент текста по ключевому слову. Также в наборе есть более общие вопросы, например про отличия ПАД от других программ или про карьерные возможности.

## 5. Создание ChromaDB и загрузка чанков

In [57]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

client = chromadb.Client()

try:
    client.delete_collection(name='dsba_docs')
except:
    pass

collection = client.create_collection(name='dsba_docs')

embeddings = model.encode(chunks).tolist()

ids = []

for i in range(len(chunks)):
    ids.append("chunk_" + str(i))

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print(f'Загружено {collection.count()} чанков в ChromaDB')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Загружено 67 чанков в ChromaDB


## 6. Проверка поиска на одном вопросе

In [58]:
question = "Какие предметы ЕГЭ нужны для поступления на ПАД?"

question_embedding = model.encode(question).tolist()

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print("Вопрос:")
print(question)

print("\nНайденные чанки:")

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Чанк {i + 1} ---")
    print(doc[:700])

Вопрос:
Какие предметы ЕГЭ нужны для поступления на ПАД?

Найденные чанки:

--- Чанк 1 ---
Для поступающих: Траектория поступления Программа «Прикладной анализ данных» реализуется на факультете компьютерных наук и предлагает абитуриентам 220 платных мест и 10 платных мест для иностранных студентов (бюджетные места не предусмотрены). Скидки поступающим Для абитуриентов программы действует система скидок по результатам поступления, отличная от общеуниверситетской. Положение о скидках для поступающих в 2025 году будет опубликовано весной 2025. О скидках в формате FAQ можно почитать в разделе Скидки на обучение. Поступление на программу российских абитуриентов Для поступления необходимо предоставить сведения о баллах Единого государственного экзамена (ЕГЭ) по следующим предметам: Мате

--- Чанк 2 ---
Скидки на обучение для поступивших: Кто может претендовать на скидку? Претендовать на скидку могут как абитуриенты, имеющие олимпиадные достижения, так и поступающие на основании баллов ЕГЭ. Ч

## 7. Оценка retrieval на golden set

In [59]:
results_list = []

hits_at_3 = 0
hits_at_5 = 0

for question, keyword, category in golden_set:
    question_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=5
    )

    docs = results["documents"][0]

    found_at_3 = False
    found_at_5 = False

    for doc in docs[:3]:
        if keyword.lower() in doc.lower():
            found_at_3 = True

    for doc in docs[:5]:
        if keyword.lower() in doc.lower():
            found_at_5 = True

    if found_at_3:
        hits_at_3 += 1

    if found_at_5:
        hits_at_5 += 1

    results_list.append({
        "question": question,
        "keyword": keyword,
        "category": category,
        "hit_at_3": found_at_3,
        "hit_at_5": found_at_5
    })

hit_rate_3 = hits_at_3 / len(golden_set)
hit_rate_5 = hits_at_5 / len(golden_set)

results_df = pd.DataFrame(results_list)

print(f"Hit Rate@3: {hit_rate_3:.2%}")
print(f"Hit Rate@5: {hit_rate_5:.2%}")

display(results_df.head())

Hit Rate@3: 58.00%
Hit Rate@5: 72.00%


,question,keyword,category,hit_at_3,hit_at_5
0,Какова основная цель программы ПАД?,подготовка высококвалифицированных аналитиков,program_info,True,True
1,В каком году была создана программа ПАД?,2018,program_info,True,True
2,С каким британским университетом создана программа?,LSE,program_info,True,True
3,На каком факультете реализуется программа ПАД?,факультете компьютерных наук,program_info,False,True
4,В каком году ВШЭ и Яндекс открыли факультет компьютерных наук?,2014,program_info,False,True


## 8. Анализ промахов

In [60]:
misses_df = results_df[results_df["hit_at_5"] == False]

print(f"Количество промахов Hit@5: {len(misses_df)}")

display(misses_df)

Количество промахов Hit@5: 14


,question,keyword,category,hit_at_3,hit_at_5
5,Какие специализации есть на программе?,Анализ данных в бизнесе,program_info,False,False
9,Какой минимальный балл ЕГЭ по математике?,70,admission,False,False
11,Какой минимальный балл ЕГЭ по русскому языку?,60,admission,False,False
17,Какой минимальный балл по математике для иностранцев?,75,admission,False,False
18,Какой минимальный балл по английскому для иностранцев?,70,admission,False,False
22,Какой коэффициент у балла ЕГЭ по математике при ранжировании?,коэффициент 3,tuition,False,False
31,На каком курсе изучается Машинное обучение?,Машинное обучение,curriculum,False,False
32,На каком курсе изучаются Базы данных?,Базы данных,curriculum,False,False
33,На каком курсе изучается Глубокое обучение?,Глубокое обучение,curriculum,False,False
34,На каком курсе изучается Эконометрика?,Эконометрика,curriculum,False,False


Промах означает, что нужное ключевое слово не встретилось среди top-5 найденных чанков. Это не всегда значит, что бот полностью ошибся, но показывает, что нужный фрагмент текста не был найден достаточно высоко.

Промахи происходят в основном из-за нескольких причин: слишком большой или слишком маленький размер чанка, повторяющиеся названия курсов в учебном плане.

## 9. Функция для экспериментов

In [61]:
def evaluate_chunks(chunks_for_test):
    temp_client = chromadb.Client()

    try:
        temp_client.delete_collection(name="temp_collection")
    except:
        pass

    temp_collection = temp_client.create_collection(name="temp_collection")

    temp_embeddings = model.encode(chunks_for_test).tolist()

    temp_ids = []
    for i in range(len(chunks_for_test)):
        temp_ids.append("temp_chunk_" + str(i))

    temp_collection.add(
        documents=chunks_for_test,
        embeddings=temp_embeddings,
        ids=temp_ids
    )

    temp_hits_at_3 = 0
    temp_hits_at_5 = 0

    for item in golden_set:
        question = item[0]
        keyword = item[1]

        question_embedding = model.encode(question).tolist()

        results = temp_collection.query(
            query_embeddings=[question_embedding],
            n_results=5
        )

        docs = results["documents"][0]

        found_at_3 = False
        found_at_5 = False

        first_3_docs = docs[:3]
        first_5_docs = docs[:5]

        for doc in first_3_docs:
            doc_lower = doc.lower()
            keyword_lower = keyword.lower()

            if keyword_lower in doc_lower:
                found_at_3 = True

        for doc in first_5_docs:
            doc_lower = doc.lower()
            keyword_lower = keyword.lower()

            if keyword_lower in doc_lower:
                found_at_5 = True

        if found_at_3 == True:
            temp_hits_at_3 = temp_hits_at_3 + 1

        if found_at_5 == True:
            temp_hits_at_5 = temp_hits_at_5 + 1

    temp_hit_rate_3 = temp_hits_at_3 / len(golden_set)
    temp_hit_rate_5 = temp_hits_at_5 / len(golden_set)

    return temp_hit_rate_3, temp_hit_rate_5

## 10. Эксперименты с размером чанка

In [62]:
chunk_sizes = [40, 80, 150, 500]

chunk_results = []

for size in chunk_sizes:
    chunks_for_test = split_into_chunks(text, chunk_size=size, overlap=10)

    hit_rate_3, hit_rate_5 = evaluate_chunks(chunks_for_test)

    chunk_results.append({
        "chunk_size": size,
        "overlap": 10,
        "number_of_chunks": len(chunks_for_test),
        "hit_rate_3": hit_rate_3,
        "hit_rate_5": hit_rate_5
    })

chunk_results_df = pd.DataFrame(chunk_results)

display(chunk_results_df)

,chunk_size,overlap,number_of_chunks,hit_rate_3,hit_rate_5
0,40,10,141,0.72,0.76
1,80,10,95,0.74,0.84
2,150,10,78,0.64,0.68
3,500,10,66,0.62,0.72


В этом эксперименте сравнивались разные размеры чанков. Метрики Hit Rate@3 и Hit Rate@5 показывают, как часто правильный фрагмент текста попадает в первые 3 или первые 5 найденных результатов. Слишком большие чанки могут содержать много лишней информации, поэтому поиск становится менее точным. Слишком маленькие чанки могут потерять контекст.

## 11. Эксперимент с предобработкой текста

In [63]:
raw_text = text

basic_text = text
basic_text = basic_text.replace("\n\n\n", "\n\n")
basic_text = basic_text.replace("  ", " ")

advanced_text = text
advanced_text = advanced_text.replace("\xa0", " ")
advanced_text = advanced_text.replace("–", "-")
advanced_text = advanced_text.replace("—", "-")
advanced_text = advanced_text.replace("«", '"')
advanced_text = advanced_text.replace("»", '"')
advanced_text = re.sub(r" +", " ", advanced_text)
advanced_text = re.sub(r"\n\s*\n\s*\n+", "\n\n", advanced_text)
advanced_text = re.sub(r"-{5,}", "\n" + "-" * 20 + "\n", advanced_text)

raw_chunks = split_into_chunks(raw_text, chunk_size=80, overlap=10)
basic_chunks = split_into_chunks(basic_text, chunk_size=80, overlap=10)
advanced_chunks = split_into_chunks(advanced_text, chunk_size=80, overlap=10)

raw_hit_3, raw_hit_5 = evaluate_chunks(raw_chunks)
basic_hit_3, basic_hit_5 = evaluate_chunks(basic_chunks)
advanced_hit_3, advanced_hit_5 = evaluate_chunks(advanced_chunks)

preprocessing_results = []

preprocessing_results.append({
    "variant": "raw_text",
    "number_of_chunks": len(raw_chunks),
    "hit_rate_3": raw_hit_3,
    "hit_rate_5": raw_hit_5
})

preprocessing_results.append({
    "variant": "basic_cleaning",
    "number_of_chunks": len(basic_chunks),
    "hit_rate_3": basic_hit_3,
    "hit_rate_5": basic_hit_5
})

preprocessing_results.append({
    "variant": "advanced_cleaning",
    "number_of_chunks": len(advanced_chunks),
    "hit_rate_3": advanced_hit_3,
    "hit_rate_5": advanced_hit_5
})

preprocessing_results_df = pd.DataFrame(preprocessing_results)

display(preprocessing_results_df)

,variant,number_of_chunks,hit_rate_3,hit_rate_5
0,raw_text,95,0.74,0.84
1,basic_cleaning,95,0.74,0.84
2,advanced_cleaning,95,0.76,0.84


В этом эксперименте сравнивался исходный текст и немного очищенный текст. Очистка простая: убираются лишние пробелы и слишком большие разрывы между строками. Такой эксперимент нужен, потому что качество текста влияет на качество поиска.

## 12. Сравнение эмбеддеров

In [64]:
embedding_models = [
    "paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small"
]

embedding_results = []

chunks_for_test = split_into_chunks(text, chunk_size=80, overlap=10)

for model_name in embedding_models:
    model = SentenceTransformer(model_name)

    hit_3, hit_5 = evaluate_chunks(chunks_for_test)

    embedding_results.append({
        "model": model_name,
        "chunk_size": 80,
        "hit_rate_3": hit_3,
        "hit_rate_5": hit_5
    })

embedding_results_df = pd.DataFrame(embedding_results)

display(embedding_results_df)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,chunk_size,hit_rate_3,hit_rate_5
0,paraphrase-multilingual-MiniLM-L12-v2,80,0.74,0.84
1,intfloat/multilingual-e5-small,80,0.80,0.86


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


В этом эксперименте сравнивались две модели эмбеддингов.
Для честного сравнения размер чанка был одинаковым для обеих моделей: 80 слов.

## 13. Реранкер

In [65]:
reranker_chunks = split_into_chunks(text, chunk_size=80, overlap=10)

reranker_client = chromadb.Client()

try:
    reranker_client.delete_collection(name="reranker_collection")
except:
    pass

reranker_collection = reranker_client.create_collection(name="reranker_collection")

reranker_embeddings = model.encode(reranker_chunks).tolist()

reranker_ids = []

for i in range(len(reranker_chunks)):
    reranker_ids.append("reranker_chunk_" + str(i))

reranker_collection.add(
    documents=reranker_chunks,
    embeddings=reranker_embeddings,
    ids=reranker_ids
)

reranker_hits_at_3 = 0
reranker_hits_at_5 = 0

for item in golden_set:
    question = item[0]
    keyword = item[1]

    question_embedding = model.encode(question).tolist()

    results = reranker_collection.query(
        query_embeddings=[question_embedding],
        n_results=10
    )

    docs = results["documents"][0]

    scores = []

    question_words = question.lower().split()

    for doc in docs:
        score = 0
        doc_lower = doc.lower()

        for word in question_words:
            if word in doc_lower:
                score = score + 1

        scores.append(score)

    reranked_docs = []

    for i in range(len(docs)):
        best_score = max(scores)
        best_index = scores.index(best_score)

        reranked_docs.append(docs[best_index])

        scores[best_index] = -1

    found_at_3 = False
    found_at_5 = False

    for doc in reranked_docs[:3]:
        if keyword.lower() in doc.lower():
            found_at_3 = True

    for doc in reranked_docs[:5]:
        if keyword.lower() in doc.lower():
            found_at_5 = True

    if found_at_3 == True:
        reranker_hits_at_3 = reranker_hits_at_3 + 1

    if found_at_5 == True:
        reranker_hits_at_5 = reranker_hits_at_5 + 1

reranker_hit_rate_3 = reranker_hits_at_3 / len(golden_set)
reranker_hit_rate_5 = reranker_hits_at_5 / len(golden_set)

normal_chunks = split_into_chunks(text, chunk_size=80, overlap=10)
normal_hit_3, normal_hit_5 = evaluate_chunks(normal_chunks)

reranker_results = []

reranker_results.append({
    "pipeline": "retriever_only",
    "hit_rate_3": normal_hit_3,
    "hit_rate_5": normal_hit_5
})

reranker_results.append({
    "pipeline": "retriever_plus_simple_reranker",
    "hit_rate_3": reranker_hit_rate_3,
    "hit_rate_5": reranker_hit_rate_5
})

reranker_results_df = pd.DataFrame(reranker_results)

display(reranker_results_df)

,pipeline,hit_rate_3,hit_rate_5
0,retriever_only,0.74,0.84
1,retriever_plus_simple_reranker,0.84,0.84


В этом эксперименте сравнивался обычный поиск и поиск с простым reranker. Сначала ChromaDB находил top-10 чанков, а затем reranker пересортировывал их по совпадению слов с вопросом. Возможно такой reranker считается простым, но он показывает идею второго этапа поиска: сначала найти несколько возможных фрагментов, а потом выбрать из них наиболее подходящие.

## 14. Latency: retrieval vs reranker

In [66]:
latency_questions = []

for item in golden_set:
    latency_questions.append(item[0])

start_time = time.time()

for question in latency_questions:
    question_embedding = model.encode(question).tolist()

    results = reranker_collection.query(
        query_embeddings=[question_embedding],
        n_results=5
    )

end_time = time.time()

retrieval_time = end_time - start_time
retrieval_avg_time = retrieval_time / len(latency_questions)


start_time = time.time()

for question in latency_questions:
    question_embedding = model.encode(question).tolist()

    results = reranker_collection.query(
        query_embeddings=[question_embedding],
        n_results=10
    )

    docs = results["documents"][0]
    question_words = question.lower().split()

    for doc in docs:
        score = 0
        doc_lower = doc.lower()

        for word in question_words:
            if word in doc_lower:
                score = score + 1

end_time = time.time()

reranker_time = end_time - start_time
reranker_avg_time = reranker_time / len(latency_questions)


latency_results = []

latency_results.append({
    "pipeline": "retrieval_only",
    "total_time": retrieval_time,
    "avg_time_per_question": retrieval_avg_time
})

latency_results.append({
    "pipeline": "retrieval_plus_simple_reranker",
    "total_time": reranker_time,
    "avg_time_per_question": reranker_avg_time
})

latency_results_df = pd.DataFrame(latency_results)

display(latency_results_df)

,pipeline,total_time,avg_time_per_question
0,retrieval_only,2.460892,0.049218
1,retrieval_plus_simple_reranker,2.520964,0.050419


## 15. LLM model comparison

In [67]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY is None:
    print("OpenRouter API key not found")
else:
    print("OpenRouter API key loaded")

OpenRouter API key loaded


In [68]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client created")

OpenRouter client created


In [69]:
llm_models = [
    "openrouter/free",
    "openai/gpt-4o-mini-2024-07-18"
]

llm_questions = [
    "Какие предметы ЕГЭ нужны для поступления на ПАД?",
    "Сколько стоит обучение на программе?",
    "Есть ли бюджетные места на программе?",
    "На каком языке проходит обучение?",
    "Чем ПАД отличается от ПМИ?"
]

llm_chunks = split_into_chunks(text, chunk_size=150, overlap=20)

llm_client_chroma = chromadb.Client()

try:
    llm_client_chroma.delete_collection(name="llm_collection")
except:
    pass

llm_collection = llm_client_chroma.create_collection(name="llm_collection")

llm_embeddings = model.encode(llm_chunks).tolist()

llm_ids = []

for i in range(len(llm_chunks)):
    llm_ids.append("llm_chunk_" + str(i))

llm_collection.add(
    documents=llm_chunks,
    embeddings=llm_embeddings,
    ids=llm_ids
)

llm_results = []

for llm_model_name in llm_models:
    for question in llm_questions:
        question_embedding = model.encode(question).tolist()

        search_results = llm_collection.query(
            query_embeddings=[question_embedding],
            n_results=3
        )

        docs = search_results["documents"][0]

        context = ""

        for doc in docs:
            context = context + doc + "\n\n"

        prompt = (
            "Ответь на вопрос пользователя только по контексту ниже. "
            "Если в контексте нет ответа, напиши, что информации недостаточно.\n\n"
            "Контекст:\n"
            + context
            + "\nВопрос:\n"
            + question
        )

        try:
            response = openrouter_client.chat.completions.create(
                model=llm_model_name,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )

            answer = response.choices[0].message.content

        except Exception as e:
            answer = "ERROR: " + str(e)

        llm_results.append({
            "llm_model": llm_model_name,
            "question": question,
            "chunk_size": 150,
            "retrieved_chunks": 3,
            "answer": answer
        })

llm_results_df = pd.DataFrame(llm_results)

display(llm_results_df)

,llm_model,question,chunk_size,retrieved_chunks,answer
0,openrouter/free,Какие предметы ЕГЭ нужны для поступления на ПАД?,150,3,Для поступления на программу «Прикладной анализ данных» необходимо сдать ЕГЭ по следующим предметам: \n- **Математи...
1,openrouter/free,Сколько стоит обучение на программе?,150,3,"По данным контекста, обучение на программе стоит 1 млн рублей в год."
2,openrouter/free,Есть ли бюджетные места на программе?,150,3,\nНа программу «Прикладной анализ данных» бюджетные места не предусмотрены. Всего доступно 220 платных мест для росс...
3,openrouter/free,На каком языке проходит обучение?,150,3,Обучение проходит на английском языке.
4,openrouter/free,Чем ПАД отличается от ПМИ?,150,3,"На основании контекста, ПАД (Программа прикладной экономики и анализа данных) и ПМИ (Программа по математике и инфор..."
5,openai/gpt-4o-mini-2024-07-18,Какие предметы ЕГЭ нужны для поступления на ПАД?,150,3,Для поступления на программу «Прикладной анализ данных» необходимо предоставить сведения о баллах Единого государств...
6,openai/gpt-4o-mini-2024-07-18,Сколько стоит обучение на программе?,150,3,Обучение на программе стоит 1 миллион рублей в год.
7,openai/gpt-4o-mini-2024-07-18,Есть ли бюджетные места на программе?,150,3,Бюджетные места на программе не предусмотрены.
8,openai/gpt-4o-mini-2024-07-18,На каком языке проходит обучение?,150,3,Обучение проходит на английском языке.
9,openai/gpt-4o-mini-2024-07-18,Чем ПАД отличается от ПМИ?,150,3,ПАД (Прикладная математика и данные) и ПМИ (Прикладная математика и информатика) – это программы одного направления ...


## 16. Manual LLM evaluation

In [70]:
llm_scores = []

llm_scores.append({
    "llm_model": "openrouter/free",
    "relevance_score": 4,
    "faithfulness_score": 4,
    "clarity_score": 3,
    "notes": "Answers were relevant, but sometimes too short or less stable"
})

llm_scores.append({
    "llm_model": "openai/gpt-4o-mini-2024-07-18",
    "relevance_score": 5,
    "faithfulness_score": 5,
    "clarity_score": 5,
    "notes": "Answers were clearer and more complete in this test"
})

llm_scores_df = pd.DataFrame(llm_scores)

display(llm_scores_df)

,llm_model,relevance_score,faithfulness_score,clarity_score,notes
0,openrouter/free,4,4,3,"Answers were relevant, but sometimes too short or less stable"
1,openai/gpt-4o-mini-2024-07-18,5,5,5,Answers were clearer and more complete in this test
